# **Programación**

In [21]:
# Importación de librerías
import mne
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
from scipy.stats import normaltest, f_oneway, kruskal
from IPython.display import display
import antropy as ant
from scipy.signal import coherence
from mne.decoding import CSP
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp

In [2]:
# Verificación de carpetas completas (sujetos completos)

# Carpeta de datos
ruta = 'ds004362'

# Detectar carpetas de sujetos
sujetos = [s for s in os.listdir(ruta) if s.startswith('sub-')]

print(sujetos)

['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012', 'sub-013', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018', 'sub-019', 'sub-020', 'sub-021', 'sub-022', 'sub-023', 'sub-024', 'sub-025', 'sub-026', 'sub-027', 'sub-028', 'sub-029', 'sub-030', 'sub-031', 'sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036', 'sub-037', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043', 'sub-044', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050', 'sub-051', 'sub-052', 'sub-053', 'sub-054', 'sub-055', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065', 'sub-066', 'sub-067', 'sub-068', 'sub-069', 'sub-070', 'sub-071', 'sub-072', 'sub-073', 'sub-074', 'sub-075', 'sub-076', 'sub-077', 'sub-078', 'sub-079', 'sub-080', 'sub-081', 'sub-082', 'sub-083', 'sub-084', 'sub-085', 'sub-086', 'sub-087', 'sub-088', 'sub-089', 'sub-090', 'sub-091'

In [3]:
# Función filtrado de señales

def filtrar(senal):
    s_filt = senal.copy().filter(8., 30., fir_window='hamming', verbose=False)

    return s_filt

In [84]:
# Función para calcular PSD por época

def calcular_psd(psd_epoca, freqs, canales, bandas):
    """
    Toma el espectro de potencia de una época y extrae la
    potencia relativa de las bandas especificadas.

    Retorna una lista plana:
    [mu_C3, mu_C4, beta_C3, beta_C4]
    """

    lista_psd = []

    # Potencia total de cada canal (1-40 Hz)
    potencia_total = psd_epoca.sum(axis=1)

    for banda_nombre, (fmin, fmax) in bandas.items():

        idx_freq = (freqs >= fmin) & (freqs <= fmax)

        # Potencia de la banda
        potencia_banda = psd_epoca[:, idx_freq].sum(axis=1)

        # Potencia relativa
        potencia_relativa = potencia_banda / potencia_total

        for i in range(len(canales)):
            lista_psd.append(potencia_relativa[i])

    return lista_psd

In [13]:
# Entropía aproximada

def calcular_entropia(datos_epoca):
    """
    Calcula la Entropía Aproximada para cada canal en una sola época.
    datos_epoca: array de numpy con forma (n_canales, n_muestras)
    Retorna: Lista con los valores de ApEn [ApEn_C3, ApEn_C4, ApEn_Cz]
    """

    entrop_aprox = []
    # Iterar sobre cada canal
    for i in range(datos_epoca.shape[0]):
        senal = datos_epoca[i, :]
        entropia = ant.app_entropy(senal, order=2)
        entrop_aprox.append(entropia)
    return entrop_aprox

In [6]:
# Coherencia

def calcular_coherencia_banda(datos_epoca, sfreq, fmin, fmax):
    """
    Calcula el promedio de la coherencia en la banda Mu (8-13 Hz) entre C3 y C4.
    datos_epoca: array de numpy (n_canales, n_muestras). Asumimos 0=C3, 1=C4.
    sfreq: Frecuencia de muestreo.
    Retorna: Un solo valor flotante (Coherencia promedio)
    """

    # Asumimos que C3 es índice 0 y C4 es índice 1
    senal_c3 = datos_epoca[0, :]
    senal_c4 = datos_epoca[1, :]
    
    f, Cxy = coherence(senal_c3, senal_c4, fs=sfreq, nperseg=int(sfreq*2))
    idx_banda = np.where((f >= fmin) & (f <= fmax))[0]
    
    return np.mean(Cxy[idx_banda])

In [7]:
# CSP

def extraer_caracteristicas_csp(epochs_totales, epochs_izq, epochs_der, n_componentes=2):
    """
    Aplica CSP a las épocas de mano izquierda y derecha para extraer sus características (log-varianza).
    Retorna: Matriz de características X_csp y el vector de etiquetas Y.
    """

    # 1. Unir datos para ENTRENAR el CSP (solo Izquierda y Derecha)
    datos_izq = epochs_izq.get_data(copy=True)
    datos_der = epochs_der.get_data(copy=True)
    X_train = np.concatenate([datos_izq, datos_der])
    Y_train = np.concatenate([np.zeros(len(datos_izq)), np.ones(len(datos_der))])
    
    # 2. Entrenar CSP (fit)
    csp = CSP(n_components=n_componentes, reg=None, log=True, norm_trace=False)
    csp.fit(X_train, Y_train)
    
    # 3. Extraer características para TODAS las épocas (transform), incluido el reposo
    X_all = epochs_totales.get_data(copy=True)
    X_caracteristicas = csp.transform(X_all)
    
    return X_caracteristicas

In [85]:
# Función principal: Procesamiento de señales EEG

def procesamiento_eeg(ruta, inicio, fin, archivo_salida="resultados_caracteristicas.csv"):
    
    sujetos = [s for s in os.listdir(ruta) if s.startswith('sub-')]
    sujetos_rango = sujetos[inicio:fin]
    
    if os.path.exists(archivo_salida):
        df_total = pd.read_csv(archivo_salida)
        sujetos_procesados = set(df_total["sujeto"].unique())
        print(f"Archivo existente cargado ({len(sujetos_procesados)} sujetos ya procesados)")
    else:
        df_total = pd.DataFrame()
        sujetos_procesados = set()
    
    for sujeto in sujetos_rango:
        
        if sujeto in sujetos_procesados:
            print(f"{sujeto} ya procesado, se omite")
            continue
            
        print(f"\nProcesando {sujeto}...")
        
        ruta_sujeto = os.path.join(ruta, sujeto, "eeg")
        archivos = [f for f in os.listdir(ruta_sujeto) if f.endswith('.set')]
        runs_imag = ['run-4', 'run-8', 'run-12']
        raws_imag = []

        for archivo in archivos:
            ruta_archivo = os.path.join(ruta_sujeto, archivo)
            if any(run in archivo for run in runs_imag):
                raw = mne.io.read_raw_eeglab(ruta_archivo, preload=True, verbose=False)
                raws_imag.append(raw)

        raw_imag = mne.concatenate_raws(raws_imag)

        # Selección de canales
        canales = ['C3', 'C4']
        raw_imag.pick_channels(canales)
        
        # Filtrado base
        imag_filtrado = filtrar(raw_imag)
        
        events_i, event_id_i = mne.events_from_annotations(imag_filtrado, verbose=False)
        epochs_imag = mne.Epochs(imag_filtrado, events_i, event_id=event_id_i, tmin=0, tmax=4, 
                                 baseline=None, preload=True, verbose=False)
        
        sfreq = raw_imag.info['sfreq']
        
        bandas = {
            "mu": (8, 12),
            "beta": (13, 30)
        }
        
        # Copias filtradas para las otras métricas
        epochs_mu = epochs_imag.copy().filter(bandas["mu"][0], bandas["mu"][1], verbose=False)
        epochs_beta = epochs_imag.copy().filter(bandas["beta"][0], bandas["beta"][1], verbose=False)
        
        csp_mu = extraer_caracteristicas_csp(epochs_mu, epochs_mu['TASK2T1'], epochs_mu['TASK2T2'], n_componentes=2)
        csp_beta = extraer_caracteristicas_csp(epochs_beta, epochs_beta['TASK2T1'], epochs_beta['TASK2T2'], n_componentes=2)
        
        # CÁLCULO VECTORIZADO DE PSD GLOBAL
        psd_global, freqs = mne.time_frequency.psd_array_welch(
            epochs_imag.get_data(copy=True), sfreq=sfreq, fmin=1, fmax=40, verbose=False
        )

        datos_mu = epochs_mu.get_data(copy=True)
        datos_beta = epochs_beta.get_data(copy=True)
        id_a_condicion = {v: k for k, v in event_id_i.items()}

        filas_sujeto = []

        for epoca_idx in range(len(epochs_imag)):
            
            id_evento = epochs_imag.events[epoca_idx, 2]
            cond_str = id_a_condicion.get(id_evento, "")
            
            # ETIQUETAS NUMÉRICAS DIRECTAS
            if 'TASK2T0' in cond_str: tarea = 0
            elif 'TASK2T1' in cond_str: tarea = 1
            elif 'TASK2T2' in cond_str: tarea = 2
            else: continue 
            
            fila = {
                "sujeto": sujeto,
                "tarea": tarea,
                "epoca": epoca_idx
            }
            
            # =========================================================
            # A. EXTRAER PSD (Ahora proviene de una lista)
            # =========================================================
            lista_valores_psd = calcular_psd(psd_global[epoca_idx], freqs, canales, bandas)
            
            # Asignamos los valores manualmente desde la lista a la fila.
            # El orden de la lista es: [mu_C3, mu_C4, mu_Cz, beta_C3, beta_C4, beta_Cz]
            fila["PSD_mu_C3"] = lista_valores_psd[0]
            fila["PSD_mu_C4"] = lista_valores_psd[1]
            
            fila["PSD_beta_C3"] = lista_valores_psd[2]
            fila["PSD_beta_C4"] = lista_valores_psd[3]
            
            # =========================================================
            # B. Entropía Aproximada
            # =========================================================
            apen_mu = calcular_entropia(datos_mu[epoca_idx])
            apen_beta = calcular_entropia(datos_beta[epoca_idx])
            for i, ch in enumerate(canales):
                fila[f"ApEn_mu_{ch}"] = apen_mu[i]
                fila[f"ApEn_beta_{ch}"] = apen_beta[i]
                
            # =========================================================
            # C. Coherencia (C3-C4)
            # =========================================================
            fila["Coh_mu_C3_C4"] = calcular_coherencia_banda(datos_mu[epoca_idx], sfreq, bandas["mu"][0], bandas["mu"][1])
            fila["Coh_beta_C3_C4"] = calcular_coherencia_banda(datos_beta[epoca_idx], sfreq, bandas["beta"][0], bandas["beta"][1])
            
            # =========================================================
            # D. Características CSP
            # =========================================================
            fila["CSP_mu_comp1"] = csp_mu[epoca_idx, 0]
            fila["CSP_mu_comp2"] = csp_mu[epoca_idx, 1]
            fila["CSP_beta_comp1"] = csp_beta[epoca_idx, 0]
            fila["CSP_beta_comp2"] = csp_beta[epoca_idx, 1]
            
            filas_sujeto.append(fila)
            
        df_sujeto = pd.DataFrame(filas_sujeto)
        df_sujeto.to_csv(archivo_salida, mode='a', header=not os.path.exists(archivo_salida), index=False)
        
        print(f"--> {sujeto} listo.")

    print("Sujetos procesados - Proceso finalizado")

In [87]:
procesamiento_eeg(ruta, inicio=0, fin=109)


Procesando sub-001...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Computing rank from data with rank=None
    Using tolerance 1.3e-06 (2.2e-16 eps * 2 dim * 2.9e+09  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
Reducing data rank from 2 -> 2
Estimating class=0.0 covariance using EMPIRICAL
Done.
Estimating class=1.0 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 1.4e-06 (2.2e-16 eps * 2 dim * 3.1e+09  max singular value)
    Estimated rank (data): 2
    data: rank 2 computed from 2 data channels with 0 projectors
Reducing data rank from 2 -> 2
Estimating class=0.0 covariance using EMPIRICAL
Done.
Estimating class=1.0 covariance using EMPIRICAL
Done.
--> sub-001 listo.

Procesando sub-002...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Computing rank from data with rank=None
    Using tolerance 7.2e-07 (

In [88]:
df = pd.read_csv("resultados_caracteristicas.csv")
display(df.head(10))

,sujeto,tarea,epoca,PSD_mu_C3,PSD_mu_C4,PSD_beta_C3,PSD_beta_C4,ApEn_mu_C3,ApEn_beta_C3,ApEn_mu_C4,ApEn_beta_C4,Coh_mu_C3_C4,Coh_beta_C3_C4,CSP_mu_comp1,CSP_mu_comp2,CSP_beta_comp1,CSP_beta_comp2
0,sub-001,0,0,0.231152,0.348668,0.609935,0.490101,0.570400,0.886904,0.599652,0.976759,0.669231,0.609161,-1.094025,-0.793854,-1.004695,-1.088702
1,sub-001,2,1,0.286759,0.206453,0.560354,0.663636,0.578993,0.890194,0.590078,0.864570,0.646239,0.677454,-1.520772,-1.221679,-1.128982,-0.859706
2,sub-001,0,2,0.387253,0.408716,0.342264,0.325357,0.571612,0.976583,0.577775,1.051539,0.711312,0.617336,-0.589068,-0.692220,-0.671218,-1.075737
3,sub-001,1,3,0.333244,0.375185,0.513737,0.497623,0.576481,0.889757,0.586594,0.881570,0.727851,0.615471,-0.615036,-0.879434,-0.972503,-0.799456
4,sub-001,0,4,0.446755,0.535208,0.371091,0.322405,0.597163,0.911826,0.595641,0.852669,0.866652,0.606289,-0.964911,-0.618937,-0.887249,-0.786711
5,sub-001,2,5,0.339293,0.449310,0.486445,0.435230,0.567511,0.916004,0.560001,0.971846,0.735539,0.523273,-1.075399,-1.001390,-0.922628,-1.214172
6,sub-001,0,6,0.369664,0.362948,0.515876,0.485102,0.560620,0.921381,0.605512,0.959796,0.580562,0.655901,-0.420203,-0.357011,-0.524810,-0.606461
7,sub-001,1,7,0.344937,0.326295,0.531260,0.514776,0.568765,0.960377,0.544803,0.948502,0.740117,0.578058,-0.677710,-1.228096,-0.769802,-1.065826
8,sub-001,0,8,0.321571,0.386067,0.406642,0.403038,0.583924,0.872035,0.571143,0.921557,0.732227,0.666664,-0.593083,-0.858830,-0.619524,-0.905999
9,sub-001,1,9,0.567488,0.328061,0.293636,0.450087,0.551311,0.930532,0.563831,0.916404,0.754648,0.626342,-0.030796,-0.795509,-0.695216,-0.728047


# **Análisis de resultados**

### **Estadística descriptiva:**

### **Planteamiento de hipótesis:**

In [89]:
def evaluar_hipotesis(df, variable):

    print(f"Variable: {variable}")

    # Separar condiciones

    reposo = df[df["tarea"] == 0][variable].dropna()
    mi     = df[df["tarea"] == 1][variable].dropna()
    md     = df[df["tarea"] == 2][variable].dropna()

    # 1. Prueba de Normalidad

    p_rep = normaltest(reposo).pvalue
    p_mi  = normaltest(mi).pvalue
    p_md  = normaltest(md).pvalue

    normal = (p_rep > 0.05 and p_mi > 0.05 and p_md > 0.05)

    print("\nPrueba de normalidad (D'Agostino-Pearson):")

    col_norm = ["Condición", "p-valor", "Normalidad"]
    
    fil_norm = [
        ["Reposo", f"{p_rep:.4e}", "Sí" if p_rep > 0.05 else "No"],
        ["Imaginación mano izquierda", f"{p_mi:.2e}", "Sí" if p_mi > 0.05 else "No"],
        ["Imaginación mano derecha", f"{p_md:.2e}", "Sí" if p_md > 0.05 else "No"]
    ]
    
    # Resultados de normalidad
    display(pd.DataFrame(fil_norm, columns = col_norm))

    # 2. Pruebas de hipótesis

    if normal:

        stat, p = f_oneway(reposo, mi, md)
        prueba = "ANOVA"

    else:
        stat, p = kruskal(reposo, mi, md)
        prueba = "Kruskal-Wallis"

    print("\nPrueba de hipótesis: ")
    print(f"\n{prueba} ---> p-valor = {p:.4e}")

    # DECISION

    if p < 0.05:

        print("\nResultado: Se rechaza H0")

        # 3. POST-HOC

        print("\nPOST-HOC")

        if normal:

            tukey = pairwise_tukeyhsd(endog=df[variable],
                groups=df["tarea"], alpha=0.05)

            display(tukey)

        else:

            dunn = sp.posthoc_dunn(df, val_col=variable,
                group_col="tarea", p_adjust="bonferroni")

            display(dunn)

    else:

        print("\nResultado: No se rechaza H0")

### **Análisis 1:** Análisis del índice de lateralización

- **Objetivo:** Evaluar si el índice de lateralización de la potencia espectral en las bandas μ y β presentan diferencias entre las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha, con el fin de determinar su utilidad para la discriminación de estados motores.

    Se define el índice de lateralización como:

<center>

$$
LI = PSD,C3 −PSD,C4
$$

</center>


#### 1. **Banda μ (8–12 Hz)**

**$H_0$:**

La distribución del índice de lateralización de la potencia espectral en la banda μ es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución del índice de lateralización de la potencia espectral en la banda μ difiere en al menos una de las condiciones evaluadas.

#### 2. **Banda β (13–30 Hz)**

**$H_0$:**

La distribución del índice de lateralización de la potencia espectral en la banda β es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución del índice de lateralización de la potencia espectral en la banda β difiere en al menos una de las condiciones evaluadas.

In [90]:
df["PSD_mu_lat"] = df["PSD_mu_C3"] - df["PSD_mu_C4"]

df["PSD_beta_lat"] = df["PSD_beta_C3"] - df["PSD_beta_C4"]

In [91]:
evaluar_hipotesis(df, "PSD_mu_lat")

Variable: PSD_mu_lat

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,4.7251e-31,No
1,Imaginación mano izquierda,6.90e-24,No
2,Imaginación mano derecha,9.10e-23,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 3.4443e-03

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000,0.108548,0.220052
1,0.108548,1.000000,0.002323
2,0.220052,0.002323,1.000000


In [92]:
evaluar_hipotesis(df, "PSD_beta_lat")

Variable: PSD_beta_lat

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,2.3043e-17,No
1,Imaginación mano izquierda,7.27e-20,No
2,Imaginación mano derecha,8.87e-06,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 1.0103e-03

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000,0.011643,0.641478
1,0.011643,1.000000,0.001068
2,0.641478,0.001068,1.000000


### **Análisis 2:** Análisis de la lateralización de la entropía aproximada (ApEn)

- **Objetivo:** Analizar si la lateralización de la entropía aproximada en las bandas μ y β presenta diferencias entre las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

    Se define el índice de lateralización como:

<center>

$$
ΔApEn ​= ApEn,C3​ − ApEn,C4
$$

</center>


#### 1. **Banda μ (8–12 Hz)**

**$H_0$:**

La distribución de la lateralización de la entropía aproximada en la banda μ es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución de la lateralización de la entropía aproximada en la banda μ difiere en al menos una de las condiciones evaluadas.

#### 2. **Banda β (13–30 Hz)**

**$H_0$:**

La distribución de la lateralización de la entropía aproximada en la banda β es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución de la lateralización de la entropía aproximada en la banda β difiere en al menos una de las condiciones evaluadas.

In [94]:
df["ApEn_mu_lat"] = df["ApEn_mu_C3"] - df["ApEn_mu_C4"]

df["ApEn_beta_lat"] = df["ApEn_beta_C3"] - df["ApEn_beta_C4"]

In [95]:
evaluar_hipotesis(df, "ApEn_mu_lat")

Variable: ApEn_mu_lat

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,3.0273e-02,No
1,Imaginación mano izquierda,6.60e-01,Sí
2,Imaginación mano derecha,3.51e-02,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 7.0953e-01

Resultado: No se rechaza H0


In [96]:
evaluar_hipotesis(df, "ApEn_beta_lat")

Variable: ApEn_beta_lat

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,1.4896e-19,No
1,Imaginación mano izquierda,5.94e-05,No
2,Imaginación mano derecha,3.92e-11,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 6.8734e-01

Resultado: No se rechaza H0


### **Análisis 3:** Análisis de la coherencia

- **Objetivo:** Evaluar si los valores de coherencia entre los canales C3 y C4 en las bandas μ y β presentan diferencias entre las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

#### 1. **Banda μ (8–12 Hz)**

**$H_0$:**

La distribución de la coherencia C3–C4 en la banda μ es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución de la coherencia C3–C4 en la banda μ difiere en al menos una de las condiciones evaluadas.

#### 2. **Banda β (13–30 Hz)**

**$H_0$:**

La distribución de la coherencia C3–C4 en la banda β es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución de la coherencia C3–C4 en la banda β difiere en al menos una de las condiciones evaluadas.

In [97]:
evaluar_hipotesis(df, "Coh_mu_C3_C4")

Variable: Coh_mu_C3_C4

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,5.3722e-34,No
1,Imaginación mano izquierda,2.91e-22,No
2,Imaginación mano derecha,5.04e-23,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 2.3640e-12

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000e+00,8.516436e-10,6.617697e-08
1,8.516436e-10,1.000000e+00,1.000000e+00
2,6.617697e-08,1.000000e+00,1.000000e+00


In [98]:
evaluar_hipotesis(df, "Coh_beta_C3_C4")

Variable: Coh_beta_C3_C4

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,7.5206e-07,No
1,Imaginación mano izquierda,1.80e-01,Sí
2,Imaginación mano derecha,3.97e-02,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 3.6005e-19

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000e+00,7.717282e-17,1.217612e-09
1,7.717282e-17,1.000000e+00,1.809321e-01
2,1.217612e-09,1.809321e-01,1.000000e+00


### **Análisis 4:** Análisis de los Patrones Espaciales Comunes (CSP)

- **Objetivo:** Analizar si los Patrones Espaciales Comunes (CSP) obtenidos en las bandas μ y β presentan diferencias entre las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

#### 1. **Banda μ (8–12 Hz)**

**$H_0$:**

La distribución de los componentes CSP de la banda μ es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución de los componentes CSP de la banda μ difiere en al menos una de las condiciones evaluadas.

#### 2. **Banda β (13–30 Hz)**

**$H_0$:**

La distribución de los componentes CSP de la banda β es igual en las condiciones de reposo, imaginación de movimiento de la mano izquierda e imaginación de movimiento de la mano derecha.

**$H_1$:**

La distribución de los componentes CSP de la banda β difiere en al menos una de las condiciones evaluadas.

In [99]:
evaluar_hipotesis(df, "CSP_mu_comp1")

Variable: CSP_mu_comp1

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,6.0566e-23,No
1,Imaginación mano izquierda,3.65e-17,No
2,Imaginación mano derecha,4.98e-11,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 2.1337e-59

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000e+00,9.563167e-56,4.625536e-21
1,9.563167e-56,1.000000e+00,2.491834e-07
2,4.625536e-21,2.491834e-07,1.000000e+00


In [100]:
evaluar_hipotesis(df, "CSP_mu_comp2")

Variable: CSP_mu_comp2

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,1.6342e-30,No
1,Imaginación mano izquierda,7.17e-15,No
2,Imaginación mano derecha,2.77e-11,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 2.6699e-40

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000e+00,1.021120e-25,1.013500e-29
1,1.021120e-25,1.000000e+00,1.000000e+00
2,1.013500e-29,1.000000e+00,1.000000e+00


In [101]:
evaluar_hipotesis(df, "CSP_beta_comp1")

Variable: CSP_beta_comp1

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,4.7949e-42,No
1,Imaginación mano izquierda,2.84e-68,No
2,Imaginación mano derecha,2.08e-27,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 1.8918e-105

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000e+00,2.513526e-91,7.531787e-48
1,2.513526e-91,1.000000e+00,3.557493e-06
2,7.531787e-48,3.557493e-06,1.000000e+00


In [102]:
evaluar_hipotesis(df, "CSP_beta_comp2")

Variable: CSP_beta_comp2

Prueba de normalidad (D'Agostino-Pearson):


,Condición,p-valor,Normalidad
0,Reposo,6.0351e-45,No
1,Imaginación mano izquierda,1.19e-62,No
2,Imaginación mano derecha,3.59e-11,No



Prueba de hipótesis: 

Kruskal-Wallis ---> p-valor = 3.3445e-89

Resultado: Se rechaza H0

POST-HOC


,0,1,2
0,1.000000e+00,7.459170e-54,8.314195e-67
1,7.459170e-54,1.000000e+00,2.986801e-01
2,8.314195e-67,2.986801e-01,1.000000e+00


# **Clasificación:**

In [83]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)